In [3]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

# 데이터 경로
data_path = r"D:\02_사업관리\2025년\데이터 구매 및 활용\12_적재확인\01_생활인구\\"
output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\02_집계결과\\"

# 행정동코드 변환용 데이터프레임
code = pd.read_excel(r"D:\02_사업관리\코드자료\행정경계코드\jscode20260102\KIKcd_H.20260102.xlsx", dtype={'행정동코드': str}, engine='openpyxl')
code = code[(code['시도명']=='부산광역시') & (~code['시군구명'].isnull()) & (~code['읍면동명'].isnull())]
code = code[['행정동코드', '시군구명', '읍면동명']]
code.columns = ['hcode', 'gname', 'hname']

# 50m격자, 100m격자 불러오기
grid50 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\bigdata_50m_busan.shp").set_crs(5179)
grid100 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\grid_마라_100M.shp", encoding="utf-8").set_crs(5179)

# 사업지 영역데이터
# area = pd.read_csv(r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\\prj_area.csv", sep="|", encoding="utf-8")
# area['geometry'] = area['geometry'].apply(wkt.loads)
# area = gpd.GeoDataFrame(area, geometry='geometry', crs=5179)
            

##### 통신사 50m 성연령, 시간대 생활인구 집계

In [2]:
# 50m 성연령, 시간대 생활인구집계함수

def filtered(df):
    gdf = gpd.sjoin(area, grid50, how='inner', predicate='intersects')
    grdlist = gdf[['id','areanm']]
    df = grdlist.merge(df, how="left", on="id")
    cols = [col for col in df.columns if 'h_' in col or 'w_' in col or 'v_' in col or '_t_' in col]
    df[cols] = df[cols].astype(float)
    df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]')
    df['std_ym'] = df['std_ymd'].astype(str).str[:7]
    return df

def genders(df):
    df = filtered(df)
    
    gender_dict = {'m_sum': [col for col in df.columns if '_m_' in col], 'w_sum': [col for col in df.columns if '_w_' in col]}
    for key, value in gender_dict.items():
        df[key] = df.loc[:, value].sum(axis=1)
    group = round(df.groupby(['std_ym','id'], as_index=False)[['m_sum', 'w_sum']].sum()).sort_values(['std_ym'], ascending=True)
    group = group.melt(id_vars=['std_ym', 'id'], value_vars=['m_sum', 'w_sum'], var_name='class', value_name='pop')
    group.loc[group['class'].str.startswith('m_'), 'gender'] = '남성'
    group.loc[group['class'].str.startswith('w_'), 'gender'] = '여성'
    group = group[['std_ym', 'id', 'gender', 'pop']]
    group.columns = ['기준연월', 'id', '성별', '인구수']
    
    return group

def ages(df):
    df = filtered(df)
    
    ages_dict = {
        '20le_sum': [col for col in df.columns if col[-4:] in ['0009','1014','1519']], '20eq_sum': [col for col in df.columns if col[-4:] in ['2024','2529']],
        '30eq_sum': [col for col in df.columns if col[-4:] in ['3034','3539']], '40eq_sum': [col for col in df.columns if col[-4:] in ['4044','4549']],
        '50eq_sum': [col for col in df.columns if col[-4:] in ['5054','5559']], '60gt_sum': [col for col in df.columns if col[-4:] in ['6064','6569','7000']],}
    for key, value in ages_dict.items():
        df[key] = df.loc[:, value].sum(axis=1)
    group = round(df.groupby(['std_ym','id'], as_index=False)[['20le_sum', '20eq_sum', '30eq_sum', '40eq_sum', '50eq_sum', '60gt_sum']].sum()).sort_values(['std_ym'], ascending=True)
    group = group.melt(id_vars=['std_ym', 'id'], value_vars=['20le_sum', '20eq_sum', '30eq_sum', '40eq_sum', '50eq_sum', '60gt_sum'], var_name='class', value_name='pop')
    group.loc[group['class'].str.startswith('20le'), 'age'] = '20대미만'
    group.loc[group['class'].str.startswith('20eq'), 'age'] = '20대'
    group.loc[group['class'].str.startswith('30eq'), 'age'] = '30대'
    group.loc[group['class'].str.startswith('40eq'), 'age'] = '40대'
    group.loc[group['class'].str.startswith('50eq'), 'age'] = '50대'
    group.loc[group['class'].str.startswith('60gt'), 'age'] = '60대이상'
    group = group[['std_ym', 'id', 'age', 'pop']]
    group.columns = ['기준연월', 'id', '연령', '인구수']
    
    return group

def times(df):
    df = filtered(df)
    
    times_dict = {
        '00시': [col for col in df.columns if col[-2:] in '00'], '01시': [col for col in df.columns if col[-2:] in '01'], '02시': [col for col in df.columns if col[-2:] in '02'],
        '03시': [col for col in df.columns if col[-2:] in '03'], '04시': [col for col in df.columns if col[-2:] in '04'], '05시': [col for col in df.columns if col[-2:] in '05'],
        '06시': [col for col in df.columns if col[-2:] in '06'], '07시': [col for col in df.columns if col[-2:] in '07'], '08시': [col for col in df.columns if col[-2:] in '08'],
        '09시': [col for col in df.columns if col[-2:] in '09'], '10시': [col for col in df.columns if col[-2:] in '10'], '11시': [col for col in df.columns if col[-2:] in '11'],
        '12시': [col for col in df.columns if col[-2:] in '12'], '13시': [col for col in df.columns if col[-2:] in '13'], '14시': [col for col in df.columns if col[-2:] in '14'],
        '15시': [col for col in df.columns if col[-2:] in '15'], '16시': [col for col in df.columns if col[-2:] in '16'], '17시': [col for col in df.columns if col[-2:] in '17'],
        '18시': [col for col in df.columns if col[-2:] in '18'], '19시': [col for col in df.columns if col[-2:] in '19'], '20시': [col for col in df.columns if col[-2:] in '20'],
        '21시': [col for col in df.columns if col[-2:] in '21'], '22시': [col for col in df.columns if col[-2:] in '22'], '23시': [col for col in df.columns if col[-2:] in '23'],}
    for key, value in times_dict.items():
        df[key] = df.loc[:, value].sum(axis=1) / 3
    keys = [key for key in times_dict.keys()]
    group = round(df.groupby(['std_ym', 'id'], as_index=False)[keys].sum()).sort_values(['std_ym'], ascending=True)
    group = group.melt(id_vars=['std_ym', 'id'], value_vars=keys, var_name='class', value_name='pop')
    group.columns = ['기준연월', 'id', '시간', '인구수']
    
    return group

In [ ]:
# 대중교통 이용현황: 생활인구 추가집계(202301, 202512)
emd_p_202301 = pd.read_csv(data_path + "kt_d_living_emd_p_202301.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
tme_p_202301 = pd.read_csv(data_path + "kt_d_living_tme_p_202301.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
emd_p_202512 = pd.read_csv(data_path + "kt_d_living_emd_p_202512.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
tme_p_202501 = pd.read_csv(data_path + "kt_d_living_tme_p_202501.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
tme_p_202512 = pd.read_csv(data_path + "kt_d_living_tme_p_202512.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
gender_202301 = genders(emd_p_202301).to_csv(output_path + "사업지_50m_월별 성별 생활인구_202301.csv", sep="|", encoding="utf-8", index=False)
age_202301 = ages(emd_p_202301).to_csv(output_path + "사업지_50m_월별 연령 생활인구_202301.csv", sep="|", encoding="utf-8", index=False)
tme_202301 = times(tme_p_202301).to_csv(output_path + "사업지_50m_월별 시간 생활인구_202301.csv", sep="|", encoding="utf-8", index=False)
gender_202512 = genders(emd_p_202512).to_csv(output_path + "사업지_50m_월별 성별 생활인구_202512.csv", sep="|", encoding="utf-8", index=False)
age_202512 = ages(emd_p_202512).to_csv(output_path + "사업지_50m_월별 연령 생활인구_202512.csv", sep="|", encoding="utf-8", index=False)
tme_202512 = times(tme_p_202512).to_csv(output_path + "사업지_50m_월별 시간 생활인구_202512.csv", sep="|", encoding="utf-8", index=False)

In [3]:
tme_p_202501 = pd.read_csv(data_path + "kt_d_living_tme_p_202501.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
tme_202501 = times(tme_p_202501).to_csv(output_path + "사업지_50m_월별 시간 생활인구_202501.csv", sep="|", encoding="utf-8", index=False)

#### 연도별 구군 성별 장래인구 추계자료와 생활인구(방문추가) 시각화

In [5]:
import pandas as pd
import plotly.express as px

data_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\02_집계결과\1_생활유동인구 변화추이\\"
output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\"

# 1. 데이터 불러오기
future_pop = pd.read_csv(data_path + '연도 구군 성별 장래인구 현황.csv')
living_pop = pd.read_csv(data_path + '월별 구군 행정동별 성별현황(방문추가).csv')

# 2. 구군명(gname)과 구군코드(gcode) 매핑 정보 추출 (정렬 기준 생성)
gcode_mapping = future_pop[['gname', 'gcode']].drop_duplicates()
district_order = gcode_mapping.sort_values('gcode')['gname'].tolist()

# 3. 데이터 전처리 - 장래인구 추계
future_pop['gender'] = future_pop['gender'].map({'남자': '남성', '여자': '여성'})
future_agg = future_pop.groupby(['year', 'gname', 'gender'])['future_pop'].sum().reset_index()
future_agg = future_agg.merge(gcode_mapping, on='gname', how='left')
future_agg.rename(columns={'future_pop': 'population'}, inplace=True)
future_agg['pop_type'] = '장래인구 추계'

# 4. 데이터 전처리 - 생활인구 (월평균 산출)
living_pop['year'] = living_pop['std_ym'].str.split('-').str[0].astype(int)
living_agg = living_pop.groupby(['year', 'gname', '성별'])['avg_pop'].sum().reset_index()
# 연간 합계 / 12 -> 월평균 산출
living_agg['population'] = (living_agg['avg_pop'] / 12)
living_agg.rename(columns={'성별': 'gender'}, inplace=True)
living_agg = living_agg.merge(gcode_mapping, on='gname', how='left')
living_agg = living_agg[['year', 'gname', 'gcode', 'gender', 'population']]
living_agg['pop_type'] = '생활인구 (연평균)'

# 5. 데이터 통합 및 단위 변환 (천명 단위 + 정수형)
combined_df = pd.concat([future_agg, living_agg], axis=0)
# 인구수를 천명 단위로 변환하고 정수형(반올림)으로 표시
combined_df['population_k'] = (combined_df['population'] / 1000).round(0).astype(int)

# 6. for문을 이용한 연도별 시각화 생성 (디자인 변경 적용)
years = sorted(combined_df['year'].unique())

for year in years:
    df_year = combined_df[combined_df['year'] == year].copy()
    
    # Plotly 막대 차트 생성 (plotly_white 템플릿 사용)
    fig = px.bar(
        df_year,
        x="gname",
        y="population_k",
        color="gender",
        facet_row="pop_type",
        category_orders={"gname": district_order}, # gcode 기준 정렬
        title=f"📊 {year}년 구군별 장래인구 추계 vs 생활인구(연평균) 비교 (단위: 천명)",
        labels={
            "gname": "구군",
            "population_k": "인구 수 (천명)",
            "gender": "성별",
            "pop_type": "구분"
        },
        color_discrete_map={'남성': '#3498db', '여성': '#e74c3c'},
        text_auto=True,
        template='plotly_white' # 흰색 배경 템플릿의 기본형
    )
    
    # 디자인 상세 설정: 흰색 배경 및 회색 그리드
    fig.update_layout(
        height=800,
        width=1200,
        barmode='stack', # 성별 누적 막대
        xaxis_tickangle=45,
        legend_title_text='성별',
        margin=dict(t=80, b=100, l=80, r=40),
        plot_bgcolor='white',  # 차트 내부 배경색: 흰색
        paper_bgcolor='white'  # 차트 외부 배경색: 흰색
    )
    
    # X축 그리드 및 선 설정
    fig.update_xaxes(
        showgrid=True, 
        gridcolor='lightgray', # 회색 그리드 적용
        linecolor='black',     # 축 선 색상: 검정
        mirror=True            # 축 선을 반대편에도 표시
    )
    
    # Y축 그리드 및 선 설정
    fig.update_yaxes(
        showgrid=True, 
        gridcolor='lightgray', # 회색 그리드 적용
        linecolor='black',     # 축 선 색상: 검정
        mirror=True,
        title_text="인구 수 (천명)"
    )
    
    # 서브타이틀(facet) 라벨 텍스트 크기 및 정리
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_size=14))
    
    # HTML 파일 저장
    output_file = output_path + f"{year}년 구군 성별 장래인구와 생활인구(방문추가) 현황비교.html"
    fig.write_html(output_file)
    
    # 차트 표시
    fig.show()
    print(f"✅ {year}년 차트 생성 완료: {output_file}")

✅ 2023년 차트 생성 완료: D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\2023년 구군 성별 장래인구와 생활인구(방문추가) 현황비교.html


✅ 2024년 차트 생성 완료: D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\2024년 구군 성별 장래인구와 생활인구(방문추가) 현황비교.html


✅ 2025년 차트 생성 완료: D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\2025년 구군 성별 장래인구와 생활인구(방문추가) 현황비교.html


#### 연도 구군 연령 장래인구와 생활인구(방문추가) 시각화

In [6]:
import pandas as pd
import plotly.express as px

output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\"

# 1. 데이터 불러오기
future_age_df = pd.read_csv(data_path + '연도 구군 연령 장래인구 현황.csv')
living_age_df = pd.read_csv(data_path + '월별 구군 행정동별 연령현황(방문추가).csv')

# 2. 정렬 및 매핑 정보 추출 (gcode 기준 구군 정렬)
gcode_mapping = future_age_df[['gname', 'gcode']].drop_duplicates()
district_order = gcode_mapping.sort_values('gcode')['gname'].tolist()

# 데이터에 존재하는 연령분류 그대로 유지 (정렬 순서 설정)
age_category_order = ['20대미만', '20대', '30대', '40대', '50대', '60대', '70대이상']

# 3. 장래인구 추계 데이터 전처리
future_age_agg = future_age_df.groupby(['year', 'gname', 'gcode', 'ages'])['future_pop'].sum().reset_index()
future_age_agg.rename(columns={'future_pop': 'population', 'ages': '연령'}, inplace=True)
future_age_agg['pop_type'] = '장래인구 추계'

# 4. 생활인구 데이터 전처리 (월평균 산출)
# 날짜 컬럼에서 연도 추출
living_age_df['year'] = living_age_df['std_ym'].str.split('-').str[0].astype(int)
living_age_agg = living_age_df.groupby(['year', 'gname', 'gcode', '연령'])['avg_pop'].sum().reset_index()

# 요청 사항: 연도별 합계에서 12를 나누어 월평균 산출
living_age_agg['population'] = living_age_agg['avg_pop'] / 12
living_age_agg['pop_type'] = '생활인구 (연평균)'
living_age_agg = living_age_agg[['year', 'gname', 'gcode', '연령', 'population', 'pop_type']]

# 5. 데이터 통합 및 단위 변환
combined_age_df = pd.concat([future_age_agg, living_age_agg], axis=0)

# 인구수를 천명 단위로 변환하고 정수형으로 표시 (반올림 적용)
combined_age_df['population_k'] = (combined_age_df['population'] / 1000).round(0).astype(int)

# 6. for문을 이용한 연도별 시각화 생성 (디자인 적용)
years = sorted(combined_age_df['year'].unique())

for year in years:
    # 해당 연도 데이터 필터링
    df_year = combined_age_df[combined_age_df['year'] == year].copy()
    
    # Plotly 막대 차트 생성 (연령분류 유지)
    fig = px.bar(
        df_year,
        x="gname",
        y="population_k",
        color="연령",             # 연령분류(7개 카테고리) 그대로 유지
        facet_row="pop_type",      # 장래인구 vs 생활인구 상하 배치
        category_orders={
            "gname": district_order,     # X축: gcode 순서 정렬
            "연령": age_category_order      # 범례: 연령대 순서 고정
        },
        title=f"📊 {year}년 구군별 장래인구 추계 vs 생활인구(연평균) 비교 (단위: 천명)",
        labels={
            "gname": "구군",
            "population_k": "인구 수 (천명)",
            "연령": "연령대",
            "pop_type": "구분"
        },
        color_discrete_sequence=px.colors.qualitative.Prism, # 다양한 연령 구분을 위한 팔레트
        text_auto=True,             # 막대 위 정수값 표시
        template='plotly_white'      # 흰색 배경 디자인
    )
    
    # 상세 디자인 설정: 흰색 배경 및 회색 그리드
    fig.update_layout(
        height=850,
        width=1400,
        barmode='stack',             # 누적 막대 형식
        xaxis_tickangle=45,
        legend_title_text='연령대',
        margin=dict(t=100, b=120, l=80, r=40),
        plot_bgcolor='white',        # 차트 배경 흰색
        paper_bgcolor='white'
    )
    
    # 그리드 선 설정 (회색 그리드 및 검정 축선)
    fig.update_xaxes(showgrid=True, gridcolor='lightgray', linecolor='black', mirror=True)
    fig.update_yaxes(showgrid=True, gridcolor='lightgray', linecolor='black', mirror=True, title_text="인구 수 (천명)")
    
    # Facet 라벨(구분=...) 텍스트 깔끔하게 정리
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_size=14))
    
    # HTML 파일 저장
    output_file = output_path + f"{year}년 구군 연령 장래인구와 생활인구(방문추가).html"
    fig.write_html(output_file)
    
    # 차트 표시
    fig.show()
    print(f"✅ {year}년 분석 리포트 생성 완료: {output_file}")

✅ 2023년 분석 리포트 생성 완료: D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\2023년 구군 연령 장래인구와 생활인구(방문추가).html


✅ 2024년 분석 리포트 생성 완료: D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\2024년 구군 연령 장래인구와 생활인구(방문추가).html


✅ 2025년 분석 리포트 생성 완료: D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\2025년 구군 연령 장래인구와 생활인구(방문추가).html


In [2]:
import pandas as pd
import calendar
import warnings
warnings.filterwarnings('ignore')

res = []
for year in [2023, 2024, 2025]:
    for month in range(1, 13): 
        std_ym = str(year) + '-' + str(f"{month:02d}")
        daycnt = calendar.monthrange(year, month)[1]
        res.append({'std_ym': std_ym, 'daycnt': daycnt})

# 행정동코드 변환용 데이터프레임
code = pd.read_excel(r"D:\02_사업관리\코드자료\행정경계코드\jscode20260102\\KIKcd_H.20260102.xlsx", dtype={'행정동코드': str}, engine='openpyxl')

# 행정동 코드 전처리
inflow_code = code.loc[(code['행정동코드'].str[:2] != '26') & (~code['시도명'].str.contains('출장소')) & (code['시군구명'].isna()) & (code['읍면동명'].isna())]
inflow_code = inflow_code[['행정동코드', '시도명']]
inflow_code.columns = ['ccode', 'cname']

daycnt = pd.DataFrame(res).sort_values(['std_ym','daycnt']).reset_index(drop=None)

data_path1 = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\02_집계결과\2_타시도유출입 인구현황\\"
data_path2 = r"D:\02_사업관리\적재확인\01_생활인구\\"

emd_in1 = pd.read_csv(data_path1 + "일별 행정동 유입지별 성연령 생활인구.csv", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str},  encoding="utf-8")
emd_in10 = pd.read_csv(data_path2 + "kt_d_living_emd_inflow_202510.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str},  encoding="utf-8")
emd_in12 = pd.read_csv(data_path2 + "kt_d_living_emd_inflow_202512.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str},  encoding="utf-8")
emd_in1 = pd.concat([emd_in1, emd_in10], ignore_index=True)
emd_in = pd.concat([emd_in1, emd_in12], ignore_index=True)
emd_out1 = pd.read_csv(data_path1 + "일별 행정동 유출지별 성연령 생활인구.csv", dtype={'std_ymd': str, 'hcode': str, 'outflow_cd': str},  encoding="utf-8")
emd_out12 = pd.read_csv(data_path2 + "kt_d_living_emd_outflow_202512.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'outflow_cd': str},  encoding="utf-8")
emd_out = pd.concat([emd_out1, emd_out12], ignore_index=True)
tme_in1 = pd.read_csv(data_path1 + "일별 행정동 유입지별 시간대 생활인구.csv", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str}, encoding="utf-8")
tme_in12 = pd.read_csv(data_path2 + "kt_d_living_tme_inflow_202512.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str}, encoding="utf-8")
tme_in = pd.concat([tme_in1, tme_in12], ignore_index=True)
tme_out1 = pd.read_csv(data_path1 + "일별 행정동 유출지별 시간대 생활인구.csv", dtype={'std_ymd': str, 'hcode': str, 'outflow_cd': str}, encoding="utf-8")
tme_out12 = pd.read_csv(data_path2 + "kt_d_living_tme_outflow_202512.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'outflow_cd': str}, encoding="utf-8")
tme_out = pd.concat([tme_out1, tme_out12], ignore_index=True)

def in_filtered(df, year, hcode, inflow_cd1, inflow_cd2):
    df['std_ym'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:7]
    cond1 = df['std_ymd'].str.startswith(str(year))
    cond2 = df['hcode'].str.startswith(str(hcode))
    cond3 = df['inflow_cd'].str.startswith(str(inflow_cd1))
    cond4 = df['inflow_cd'].str.startswith(str(inflow_cd2))
    df = df.loc[cond1 & cond2 & ~cond3 & ~cond4].copy() 
    df['ccode'] = df['inflow_cd'].str[:2] + "00000000"
    return df

def out_filtered(df, year, hcode1, hcode2, outflow_cd):
    df['std_ym'] = df['std_ymd'].astype('datetime64[ns]').astype(str).str[:7]
    cond1 = df['std_ymd'].str.startswith(str(year))
    cond2 = df['outflow_cd'].str.startswith(str(outflow_cd))
    cond3 = df['hcode'].str.startswith(str(hcode1))
    cond4 = df['hcode'].str.startswith(str(hcode2))
    df = df.loc[cond1 & cond2 & ~cond3 & ~cond4].copy() 
    df['ccode'] = df['hcode'].str[:2] + "00000000"
    return df

def split_sum(df):
    # 성-연령코드 변환
    if 'gen_age' in df.columns:
        df.loc[df['gen_age'].str.contains('m_'), 'gender'] = '남성'
        df.loc[df['gen_age'].str.contains('w_'), 'gender'] = '여성'
        df.loc[df['gen_age'].str.contains('0009|1014|1519'), 'age'] = '20대미만'
        df.loc[df['gen_age'].str.contains('2024|2529'), 'age'] = '20대'
        df.loc[df['gen_age'].str.contains('3034|3539'), 'age'] = '30대'
        df.loc[df['gen_age'].str.contains('4044|4549'), 'age'] = '40대'
        df.loc[df['gen_age'].str.contains('5054|5559'), 'age'] = '50대'
        df.loc[df['gen_age'].str.contains('6064|6569'), 'age'] = '60대'
        df.loc[df['gen_age'].str.contains('7000'), 'age'] = '70대이상'
    # 시간코드 변환
    if 'tme' in df.columns:
        df.loc[(df['tme'] >= 0) & (df['tme'] <= 3), 'time'] = '01.심야(00시-03시)'
        df.loc[(df['tme'] >= 4) & (df['tme'] <= 7), 'time'] = '02.새벽(04시-07시)'
        df.loc[(df['tme'] >= 8) & (df['tme'] <= 11), 'time'] = '03.오전(08시-11시)'
        df.loc[(df['tme'] >= 12) & (df['tme'] <= 15), 'time'] = '04.오후(12시-15시)'
        df.loc[(df['tme'] >= 16) & (df['tme'] <= 19), 'time'] = '05.저녁(16시-19시)'
        df.loc[(df['tme'] >= 20) & (df['tme'] <= 23), 'time'] = '06.야간(20시-23시)'
    
    # 유입시도, 유출시도코드 변환, 강원도, 전라북도 추가
    df = pd.merge(df, inflow_code, how="left", on="ccode").sort_values(['std_ym','ccode'], ascending=True)
    df.loc[df['ccode'].str.startswith('42'), 'cname'] = '강원도'
    df.loc[df['ccode'].str.startswith('45'), 'cname'] = '전라북도'
    df['sum_pop'] = df.loc[:, ['h_pop','w_pop','v_pop']].sum(axis=1)
    return df

def in_genders(df, year, hcode, inflow_cd1, inflow_cd2):
    df = in_filtered(df, year, hcode, inflow_cd1, inflow_cd2)
    df = split_sum(df)
    group = df.groupby(['std_ym','ccode','cname','gender'], as_index=False)['sum_pop'].sum().sort_values(['std_ym','ccode'], ascending=True)
    group = group[['std_ym','ccode','cname','gender','sum_pop']]
    group.rename(columns={'sum_pop': 'sum_in_pop'}, inplace=True)
    return group

def in_ages(df, year, hcode, inflow_cd1, inflow_cd2):
    df = in_filtered(df, year, hcode, inflow_cd1, inflow_cd2)
    df = split_sum(df)
    group = df.groupby(['std_ym','ccode','cname','age'], as_index=False)['sum_pop'].sum().sort_values(['std_ym','ccode'], ascending=True)
    group = group[['std_ym','ccode','cname','age','sum_pop']]
    group.rename(columns={'sum_pop': 'sum_in_pop'}, inplace=True)
    return group

def in_times(df, year, hcode, inflow_cd1, inflow_cd2):
    df = in_filtered(df, year, hcode, inflow_cd1, inflow_cd2)
    df = split_sum(df)
    group = df.groupby(['std_ym','ccode','cname','time'], as_index=False)['sum_pop'].sum().sort_values(['std_ym','ccode'], ascending=True)
    group = group[['std_ym','ccode','cname','time','sum_pop']]
    group.rename(columns={'sum_pop': 'sum_in_pop'}, inplace=True)
    return group 

def out_genders(df, year,  hcode1, hcode2, outflow_cd):
    df = out_filtered(df, year,  hcode1, hcode2, outflow_cd)
    df = split_sum(df)
    group = df.groupby(['std_ym','ccode','cname','gender'], as_index=False)['sum_pop'].sum().sort_values(['std_ym','ccode'], ascending=True)
    group = group[['std_ym','ccode','cname','gender','sum_pop']]
    group.rename(columns={'sum_pop': 'sum_out_pop'}, inplace=True)
    return group 

def out_ages(df, year,  hcode1, hcode2, outflow_cd):
    df = out_filtered(df, year,  hcode1, hcode2, outflow_cd)
    df = split_sum(df)
    group = df.groupby(['std_ym','ccode','cname','age'], as_index=False)['sum_pop'].sum().sort_values(['std_ym','ccode'], ascending=True)
    group = group[['std_ym','ccode','cname','age','sum_pop']]
    group.rename(columns={'sum_pop': 'sum_out_pop'}, inplace=True)
    return group 

def out_times(df, year,  hcode1, hcode2, outflow_cd):
    df = out_filtered(df, year,  hcode1, hcode2, outflow_cd)
    df = split_sum(df)
    group = df.groupby(['std_ym','ccode','cname','time'], as_index=False)['sum_pop'].sum().sort_values(['std_ym','ccode'], ascending=True)
    group = group[['std_ym','ccode','cname','time','sum_pop']]
    group.rename(columns={'sum_pop': 'sum_out_pop'}, inplace=True)
    return group

# 일평균 함수 미적용
# def flowpop_mean(df):
#     df = pd.merge(df, daycnt, how="left", on="std_ym")
#     if 'gender' in df.columns  or 'age' in df.columns or 'times' in df.columns and 'sum_in_pop' in df.columns:
#         df['mean_pop'] = round(df['sum_in_pop'] / df['daycnt'])
#     elif 'gender' in df.columns or 'age' in df.columns or 'times' in df.columns and 'sum_out_pop' in df.columns:
#         df['mean_pop'] = round(df['sum_out_pop'] / df['daycnt'])
#     return df

# 2023년~2025년 성별, 연령, 시간대 유입, 유출인구 집계
ingen_lists = []
outgen_lists = []
inage_lists = []
outage_lists = []
intme_lists = []
outtme_lists = []

# for year in range(2023, 2026):
#     ingen_lists.append(in_genders(emd_in, year, 2638062, 26, 99))
#     inage_lists.append(in_ages(emd_in, year, 2638062, 26, 99))
#     intme_lists.append(in_times(tme_in, year, 2638062, 26, 99))
#     outgen_lists.append(out_genders(emd_out, year, 26, 99, 2638062))
#     outage_lists.append(out_ages(emd_out, year, 26, 99, 2638062))
#     outtme_lists.append(out_times(tme_out, year, 26, 99, 2638062))
    
# ingens = pd.concat(ingen_lists, ignore_index=True)
# ingens = ingens[['std_ym', 'cname', 'gender', 'sum_in_pop']]
# ingens.columns = ['기준연월', '유입지명', '성별', '유입인구수']
# outgens = pd.concat(outgen_lists, ignore_index=True)
# outgens = outgens[['std_ym', 'cname', 'gender', 'sum_out_pop']]
# outgens.columns = ['기준연월', '유출지명', '성별', '유출인구수']

# inages = pd.concat(inage_lists, ignore_index=True)
# inages = inages[['std_ym', 'cname', 'age', 'sum_in_pop']]
# inages.columns = ['기준연월', '유입지명', '연령', '유입인구수']
# outages = pd.concat(outage_lists, ignore_index=True)
# outages = outages[['std_ym', 'cname', 'age', 'sum_out_pop']]
# outages.columns = ['기준연월', '유출지명', '연령', '유출인구수']

# intimes = pd.concat(intme_lists, ignore_index=True)
# intimes = intimes[['std_ym', 'cname', 'time', 'sum_in_pop']]
# intimes.columns = ['기준연월', '유입지명', '시간', '유입인구수']
# outtimes = pd.concat(outtme_lists, ignore_index=True)
# outtimes = outtimes[['std_ym', 'cname', 'time', 'sum_out_pop']]
# outtimes.columns = ['기준연월', '유출지명', '시간', '유출인구수']


#### 성별 유출입인구 시각화

In [60]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import koreanize_matplotlib
output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\03_시각화결과\\"

# 성별 유출입인구 시각화 함수
def gender_flowchart(df_in, df_out):

    for df in [df_in, df_out]:
        df['연도'] = df['기준연월'].str[:4]
        df['월'] = df['기준연월'].str[-2:] + "월"

    # 권역 매핑
    kwonyeok_map = {
        '서울특별시': '수도권', '인천광역시': '수도권', '경기도': '수도권',
        '강원특별자치도': '강원권', '강원도': '강원권',
        '경상북도': '영남권', '대구광역시': '영남권',
        '전라남도': '호남권', '전라북도': '호남권', '전북특별자치도': '호남권', '광주광역시': '호남권',
        '충청남도': '충청권', '충청북도': '충청권', '대전광역시': '충청권', '세종특별자치시': '충청권',
        '울산광역시': '동남권', '경상남도': '동남권', '제주특별자치도': '제주권'
    }

    df_in['권역'] = df_in['유입지명'].map(kwonyeok_map)
    df_out['권역'] = df_out['유출지명'].map(kwonyeok_map)

    # 2. 권역별 합산 및 단위 변환 (명 -> 천명)
    df_in_grouped = df_in.groupby(['연도', '월', '성별', '권역', '기준연월'])['유입인구수'].sum().reset_index()
    df_out_grouped = df_out.groupby(['연도', '월', '성별', '권역', '기준연월'])['유출인구수'].sum().reset_index()

    df_in_grouped['유입인구수_천명'] = df_in_grouped['유입인구수'] / 1000
    df_out_grouped['유출인구수_천명'] = df_out_grouped['유출인구수'] / 1000

    # 3. Plotly 시각화 설정
    years = sorted(df_in_grouped['연도'].unique())
    genders = sorted(df_in_grouped['성별'].unique())
    n_rows = len(years) * len(genders)

    subplot_titles = []
    for y in years:
        for g in genders:
            subplot_titles.extend([f"{y}년 {g} 유입", f"{y}년 {g} 유출"])

    fig = make_subplots(
        rows=n_rows, cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.05,
        horizontal_spacing=0.08
    )

    kwonyeoks = sorted(df_in_grouped['권역'].unique())
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#1041E0']
    color_map = {k: colors[i % len(colors)] for i, k in enumerate(kwonyeoks)}

    row_idx = 1
    for year in years:
        for gender in genders:
            # 좌측: 유입 (Col 1)
            subset_in = df_in_grouped[(df_in_grouped['연도'] == year) & (df_in_grouped['성별'] == gender)].sort_values('기준연월')
            for kw in kwonyeoks:
                k_data = subset_in[subset_in['권역'] == kw]
                if not k_data.empty:
                    fig.add_trace(
                        go.Scatter(x=k_data['월'], y=k_data['유입인구수_천명'], name=kw, mode='lines+markers',
                                line=dict(color=color_map[kw]), legendgroup=kw, showlegend=(row_idx == 1),
                                hovertemplate='%{x}월 %{y:.0f}천명'),
                        row=row_idx, col=1
                    )
            
            # 우측: 유출 (Col 2)
            subset_out = df_out_grouped[(df_out_grouped['연도'] == year) & (df_out_grouped['성별'] == gender)].sort_values('기준연월')
            for kw in kwonyeoks:
                k_data = subset_out[subset_out['권역'] == kw]
                if not k_data.empty:
                    fig.add_trace(
                        go.Scatter(x=k_data['월'], y=k_data['유출인구수_천명'], name=kw, mode='lines+markers',
                                line=dict(color=color_map[kw]), legendgroup=kw, showlegend=False,
                                hovertemplate='%{x}월 %{y:.0f}천명'),
                        row=row_idx, col=2
                    )
            row_idx += 1

    # 4. 축 라벨 및 레이아웃 설정
    fig.update_layout(
        height=400 * n_rows, # 전체 행 수에 맞춰 높이 조절
        width=1400,
        title_text="연도/성별/권역별 인구 이동 추이",
        title_x=0.5,
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1)
    )

    # X축 공통 라벨 추가
    fig.update_xaxes(title_text="기준월", tickmode='linear')

    # 각 열(Column)에 맞는 Y축 라벨 개별 추가 및 정수 표시
    for i in range(1, n_rows + 1):
        fig.update_yaxes(title_text="유입인구(천명)", tickformat=".0f", row=i, col=1)
        fig.update_yaxes(title_text="유출인구(천명)", tickformat=".0f", row=i, col=2)

    # 결과 저장
    fig.write_html(output_path + '월별 성별 유출입인구 현황.html')
    fig.show()

gender_flowchart(ingens, outgens)

#### 연령 유출입인구 시각화

In [ ]:

def age_flowchart(df_in, df_out):

    for df in [df_in, df_out]:
        df['기준연월_dt'] = pd.to_datetime(df['기준연월'])
        df['연도'] = df['기준연월_dt'].dt.strftime('%Y')
        df['월'] = df['기준연월_dt'].dt.strftime('%m') + '월'

    # 권역 매핑
    kwonyeok_map = {
        '서울특별시': '수도권', '인천광역시': '수도권', '경기도': '수도권',
        '강원특별자치도': '강원권', '강원도': '강원권',
        '경상북도': '영남권', '대구광역시': '영남권',
        '전라남도': '호남권', '전라북도': '호남권', '전북특별자치도': '호남권', '광주광역시': '호남권',
        '충청남도': '충청권', '충청북도': '충청권', '대전광역시': '충청권', '세종특별자치시': '충청권',
        '울산광역시': '동남권', '경상남도': '동남권', '제주특별자치도': '제주권'
    }

    df_in['권역'] = df_in['유입지명'].map(kwonyeok_map)
    df_out['권역'] = df_out['유출지명'].map(kwonyeok_map)

    # 2. 권역별 합산 및 단위 변환 (명 -> 천명)
    df_in_grouped = df_in.groupby(['연도', '월', '연령', '권역', '기준연월'])['유입인구수'].sum().reset_index()
    df_out_grouped = df_out.groupby(['연도', '월', '연령', '권역', '기준연월'])['유출인구수'].sum().reset_index()

    df_in_grouped['유입인구수_천명'] = df_in_grouped['유입인구수'] / 1000
    df_out_grouped['유출인구수_천명'] = df_out_grouped['유출인구수'] / 1000

    # 연령대 및 월 순서 정의
    age_order = ['20대미만', '20대', '30대', '40대', '50대', '60대', '70대이상']
    month_order = [f"{i:02d}월" for i in range(1, 13)] # ['01', '02', ..., '12']

    # 3. Plotly 시각화 설정
    years = sorted(df_in_grouped['연도'].unique())
    n_rows = len(years) * len(age_order)

    subplot_titles = []
    for y in years:
        for a in age_order:
            subplot_titles.extend([f"{y}년 {a} 유입", f"{y}년 {a} 유출"])

    fig = make_subplots(
        rows=n_rows, cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.01,
        horizontal_spacing=0.08
    )

    kwonyeoks = sorted(df_in_grouped['권역'].unique())
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', "#1041E0"]
    color_map = {k: colors[i % len(colors)] for i, k in enumerate(kwonyeoks)}

    row_idx = 1
    for year in years:
        for age in age_order:
            # 좌측: 유입 (Col 1)
            subset_in = df_in_grouped[(df_in_grouped['연도'] == year) & (df_in_grouped['연령'] == age)].sort_values('기준연월')
            for kw in kwonyeoks:
                k_data = subset_in[subset_in['권역'] == kw]
                if not k_data.empty:
                    fig.add_trace(
                        go.Scatter(x=k_data['월'], y=k_data['유입인구수_천명'], name=kw, mode='lines+markers',
                                line=dict(color=color_map[kw]), legendgroup=kw, showlegend=(row_idx == 1),
                                hovertemplate='%{x}월 %{y:.0f}천명'),
                        row=row_idx, col=1
                    )
            
            # 우측: 유출 (Col 2)
            subset_out = df_out_grouped[(df_out_grouped['연도'] == year) & (df_out_grouped['연령'] == age)].sort_values('기준연월')
            for kw in kwonyeoks:
                k_data = subset_out[subset_out['권역'] == kw]
                if not k_data.empty:
                    fig.add_trace(
                        go.Scatter(x=k_data['월'], y=k_data['유출인구수_천명'], name=kw, mode='lines+markers',
                                line=dict(color=color_map[kw]), legendgroup=kw, showlegend=False,
                                hovertemplate='%{x}월 %{y:.0f}천명'),
                        row=row_idx, col=2
                    )
            row_idx += 1

    # 4. 축 라벨 및 레이아웃 설정
    fig.update_layout(
        height=350 * n_rows, # 전체 행 수에 맞춰 높이 조절
        width=1500,
        title_text="연도/연령/권역별 인구 이동 추이",
        title_x=0.5,
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.002, xanchor="right", x=1)
    )
    # 모든 X축에 대해 카테고리 순서를 01~12로 강제 지정
    fig.update_xaxes(title_text="기준월", categoryorder='array', categoryarray=month_order, tickmode='array', tickvals=month_order)
    
    # 모든 Y축에 대해 정수 눈금 고정 (dtick=1 혹은 인구수에 맞는 정수값)
    for i in range(1, n_rows + 1):
        fig.update_yaxes(title_text="유입인구(천명)", tickformat=".0f", row=i, col=1)
        fig.update_yaxes(title_text="유출인구(천명)", tickformat=".0f", row=i, col=2)

    # 결과 저장
    fig.write_html(output_path + '월별 연령 유출입인구 현황.html')
    fig.show()

age_flowchart(inages, outages)

#### 시간대 유출입인구 시각화

In [61]:
    
def time_flowchart(df_in, df_out):
    # 2. 날짜 및 연도/월 전처리
    for df in [df_in, df_out]:
        df['기준연월_dt'] = pd.to_datetime(df['기준연월'])
        df['연도'] = df['기준연월_dt'].dt.strftime('%Y')
        df['월'] = df['기준연월_dt'].dt.strftime('%m') + '월'

    # 권역 매핑 정의
    kwonyeok_map = {
        '서울특별시': '수도권', '인천광역시': '수도권', '경기도': '수도권',
        '강원특별자치도': '강원권', '강원도': '강원권',
        '경상북도': '영남권', '대구광역시': '영남권',
        '전라남도': '호남권', '전라북도': '호남권', '전북특별자치도': '호남권', '광주광역시': '호남권',
        '충청남도': '충청권', '충청북도': '충청권', '대전광역시': '충청권', '세종특별자치시': '충청권',
        '울산광역시': '동남권', '경상남도': '동남권', '제주특별자치도': '제주권'
    }

    df_in['권역'] = df_in['유입지명'].map(kwonyeok_map)
    df_out['권역'] = df_out['유출지명'].map(kwonyeok_map)

    # 3. 권역별 합산 및 단위 변환 (명 -> 천명)

    # 시간 데이터 컬럼명이 '시간대'인지 '시간'인지 확인하여 그룹화
    time_col = '시간대' if '시간대' in df_in.columns else '시간'

    # 파일에 이미 존재하는 '시간대' 컬럼을 기준으로 그룹화합니다.
    df_in_grouped = df_in.groupby(['연도', '월', time_col, '권역', '기준연월'])['유입인구수'].sum().reset_index()
    df_out_grouped = df_out.groupby(['연도', '월', time_col, '권역', '기준연월'])['유출인구수'].sum().reset_index()

    df_in_grouped['유입인구수_천명'] = df_in_grouped['유입인구수'] / 1000
    df_out_grouped['유출인구수_천명'] = df_out_grouped['유출인구수'] / 1000

    # 정렬 순서 정의
    time_order = sorted(df_in_grouped[time_col].unique())
    month_order = [f"{i:02d}월" for i in range(1, 13)]

    # 3. Plotly 시각화 설정
    years = sorted(df_in_grouped['연도'].unique())
    n_rows = len(years) * len(time_order)

    subplot_titles = []
    for y in years:
        for tg in time_order:
            subplot_titles.extend([f"{y}년 {tg} 유입", f"{y}년 {tg} 유출"])

    fig = make_subplots(
        rows=n_rows, cols=2,
        subplot_titles=subplot_titles,
        vertical_spacing=0.02,
        horizontal_spacing=0.08
    )

    # 권역별 고정 색상 설정
    kwonyeoks = sorted(df_in_grouped['권역'].unique())
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', "#1041E0"]
    color_map = {k: colors[i % len(colors)] for i, k in enumerate(kwonyeoks)}

    row_idx = 1
    for year in years:
        for tg in time_order:
            # 좌측: 유입 (Col 1)
            subset_in = df_in_grouped[(df_in_grouped['연도'] == year) & (df_in_grouped[time_col] == tg)].sort_values('기준연월')
            for kw in kwonyeoks:
                k_data = subset_in[subset_in['권역'] == kw]
                if not k_data.empty:
                    fig.add_trace(
                        go.Scatter(x=k_data['월'], y=k_data['유입인구수_천명'], name=kw, mode='lines+markers',
                                line=dict(color=color_map[kw]), legendgroup=kw, showlegend=(row_idx == 1),
                                hovertemplate='%{x}월 %{y:.0f}천명'),
                        row=row_idx, col=1
                    )
            
            # 우측: 유출 (Col 2)
            subset_out = df_out_grouped[(df_out_grouped['연도'] == year) & (df_out_grouped[time_col] == tg)].sort_values('기준연월')
            for kw in kwonyeoks:
                k_data = subset_out[subset_out['권역'] == kw]
                if not k_data.empty:
                    fig.add_trace(
                        go.Scatter(x=k_data['월'], y=k_data['유출인구수_천명'], name=kw, mode='lines+markers',
                                line=dict(color=color_map[kw]), legendgroup=kw, showlegend=False,
                                hovertemplate='%{x}월 %{y:.0f}천명'),
                        row=row_idx, col=2
                    )
            row_idx += 1

    # 5. 레이아웃 및 축 설정
    fig.update_layout(
        height=350 * n_rows, # 전체 행 수에 맞춰 높이 조절
        width=1500,
        title_text="연도/시간대/권역별 인구 이동 추이",
        title_x=0.5,
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="right", x=1)
    )

    # X축 정렬 고정 (01월~12월)
    fig.update_xaxes(title_text="기준월", categoryorder='array', categoryarray=month_order)

    # Y축 숫자 중복 방지 (dtick)
    for i in range(1, n_rows + 1):
        fig.update_yaxes(title_text="유입인구(천명)", tickformat=".0f", row=i, col=1)
        fig.update_yaxes(title_text="유출인구(천명)", tickformat=".0f", row=i, col=2)

    # 결과 저장 및 출력
    fig.write_html(output_path + '월별 시간 유출입인구 현황.html')
    fig.show()

time_flowchart(intimes, outtimes)


#### 50m 성연령 생활인구 전처리(누락분)

In [68]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

# 데이터 경로
data_path1 = r"D:\02_사업관리\적재확인\01_생활인구\\"
data_path2 = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\\"
output_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\02_집계결과\\"

emd_p = pd.read_csv(data_path1 + "kt_d_living_emd_p_202502.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
grid50 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\\bigdata_50m_busan.shp")
area = pd.read_csv(data_path2 + "prj_area.csv", sep="|", encoding="utf-8")

def geoprocess(df):
    df['geometry'] = df['geometry'].apply(wkt.loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs=5179)

def filtered(df):
    area_gdf = geoprocess(area)
    grid50['x'] = grid50.geometry.centroid.x.round(3)
    grid50['y'] = grid50.geometry.centroid.y.round(3)
    grid50_pt = gpd.GeoDataFrame(grid50, geometry=gpd.points_from_xy(grid50.x, grid50.y), crs=5179)
    gdf = gpd.sjoin(area_gdf, grid50_pt, how='inner', predicate='intersects')
    grdlist = gdf[['id','areanm']]
    df = grdlist.merge(df, how="left", on="id")
    cols = [col for col in df.columns if 'h_' in col or 'w_' in col or 'v_' in col or '_t_' in col]
    df[cols] = df[cols].astype(float)
    df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]')
    df['std_ym'] = df['std_ymd'].astype(str).str[:7]
    return df

def gender(df):
    df = filtered(df)
    gender_dict = {'m_sum': [col for col in df.columns if '_m_' in col], 'w_sum': [col for col in df.columns if '_w_' in col]}
    for key, value in gender_dict.items():
        df[key] = df.loc[:, value].sum(axis=1)
    group = round(df.groupby(['std_ym','id'], as_index=False)[['m_sum', 'w_sum']].sum()).sort_values(['std_ym'], ascending=True)
    group = group.melt(id_vars=['std_ym', 'id'], value_vars=['m_sum', 'w_sum'], var_name='class', value_name='pop')
    group.loc[group['class'].str.startswith('m_'), 'gender'] = '남성'
    group.loc[group['class'].str.startswith('w_'), 'gender'] = '여성'
    group = group[['std_ym', 'id', 'gender', 'pop']]
    group.columns = ['기준연월', 'id', '성별', '인구수']
    return group

gender = gender(emd_p)

In [69]:

emd_p = pd.read_csv(data_path1 + "kt_d_living_emd_p_202502.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'id': str}, encoding="utf-8")
grid50 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\\bigdata_50m_busan.shp")
area = pd.read_csv(data_path2 + "prj_area.csv", sep="|", encoding="utf-8")

def geoprocess(df):
    df['geometry'] = df['geometry'].apply(wkt.loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs=5179)

def filtered(df):
    area_gdf = geoprocess(area)
    grid50['x'] = grid50.geometry.centroid.x.round(3)
    grid50['y'] = grid50.geometry.centroid.y.round(3)
    grid50_pt = gpd.GeoDataFrame(grid50, geometry=gpd.points_from_xy(grid50.x, grid50.y), crs=5179)
    gdf = gpd.sjoin(area_gdf, grid50_pt, how='inner', predicate='intersects')
    grdlist = gdf[['id','areanm']]
    df = grdlist.merge(df, how="left", on="id")
    cols = [col for col in df.columns if 'h_' in col or 'w_' in col or 'v_' in col or '_t_' in col]
    df[cols] = df[cols].astype(float)
    df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]')
    df['std_ym'] = df['std_ymd'].astype(str).str[:7]
    return df

def ages(df):
    df = filtered(df)
    ages_dict = {
        '20le_sum': [col for col in df.columns if col[-4:] in ['0009','1014','1519']], 
        '20eq_sum': [col for col in df.columns if col[-4:] in ['2024','2529']],
        '30eq_sum': [col for col in df.columns if col[-4:] in ['3034','3539']],
        '40eq_sum': [col for col in df.columns if col[-4:] in ['4044','4549']],
        '50eq_sum': [col for col in df.columns if col[-4:] in ['5054','5559']], 
        '60gt_sum': [col for col in df.columns if col[-4:] in ['6064','6569','7000']],}
    for key, value in ages_dict.items():
        df[key] = df.loc[:, value].sum(axis=1)
    group = round(df.groupby(['std_ym','id'], as_index=False)[['20le_sum', '20eq_sum', '30eq_sum', '40eq_sum', '50eq_sum', '60gt_sum']].sum()).sort_values(['std_ym'], ascending=True)
    group = group.melt(id_vars=['std_ym', 'id'], value_vars=['20le_sum', '20eq_sum', '30eq_sum', '40eq_sum', '50eq_sum', '60gt_sum'], var_name='class', value_name='pop')
    group.loc[group['class'].str.startswith('20le'), 'age'] = '20대미만'
    group.loc[group['class'].str.startswith('20eq'), 'age'] = '20대'
    group.loc[group['class'].str.startswith('30eq'), 'age'] = '30대'
    group.loc[group['class'].str.startswith('40eq'), 'age'] = '40대'
    group.loc[group['class'].str.startswith('50eq'), 'age'] = '50대'
    group.loc[group['class'].str.startswith('60gt'), 'age'] = '60대이상'
    group = group[['std_ym', 'id', 'age', 'pop']]
    group.columns = ['기준연월', 'id', '연령', '인구수']
    return group

ages = ages(emd_p)

In [70]:
gender.to_csv(output_path +"사업지_50m_월별 성별 생활인구202502.csv", encoding="utf-8", index=False)
ages.to_csv(output_path + "사업지_50m_월별 연령 생활인구202502.csv", encoding="utf-8", index=False)

#### 사업지 내 월별 요일별 인구변화 추이

In [57]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from shapely import wkt

data_path = r"D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\\"

grid50 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\\bigdata_50m_busan.shp").set_crs(5179)
grid100 = gpd.read_file(r"D:\00_자료검토\03_공간자료\griddata\\grid_마라_100M.shp").set_crs(5179)
prjarea = pd.read_csv(data_path + r"02_집계결과\\prj_area.csv", sep="|", encoding="utf-8")
prjarea['geometry'] = prjarea['geometry'].apply(wkt.loads)
prjarea = gpd.GeoDataFrame(prjarea, geometry='geometry', crs=5179)

# 사업지 내 50m격자 추출
prj50m = gpd.sjoin(grid50, prjarea, how="inner", predicate="intersects")
prj50m['x'] = prj50m.geometry.centroid.x.round(3)
prj50m['y'] = prj50m.geometry.centroid.y.round(3)
prj50m_pt = gpd.GeoDataFrame(prj50m, geometry=gpd.points_from_xy(prj50m['x'], prj50m['y'], crs=5179))
prj50m_pt = prj50m_pt[['id', 'geometry']]
# print(prj50m_pt.info())
prj50m_pt = gpd.sjoin(prj50m_pt, prjarea, how="inner", predicate="intersects")
prj50m_pt = prj50m_pt[['id', 'geometry']]
# print(prj50m_pt.info())

# 사업지 내 100m격자 추출
prj100m = gpd.sjoin(grid100, prjarea, how="inner", predicate="intersects")
prj100m = prj100m[['GRID_100M_', 'geometry']]
prj100m.rename(columns={'GRID_100M_': 'grid_100m'}, inplace=True)

df = pd.read_csv(data_path + r"02_집계결과\전처리\\BUSAN_KT_T01_PRJAREA_MM.csv", sep="|", dtype={'ID': str}, encoding="utf-8").rename(columns=str.lower)
dow_dict = {'MON': '월요일', 'TUE': '화요일', 'WED': '수요일','THU': '목요일', 'FRI': '금요일', 'SAT': '토요일', 'SUN': '일요일',}
df['요일'] = df['dow'].map(dow_dict)
df.loc[df['요일'].str.contains('월요일|화요일|수요일|목요일|금요일'), '주중주말'] = '주중'
df.loc[df['요일'].str.contains('토요일|일요일'), '주중주말'] = '주말'
df['sum_pop'] = df['h_pop'] + df['w_pop'] + df['v_pop']
group = round(df.groupby(['std_ym','id','주중주말'], as_index=False)['sum_pop'].sum()).sort_values('std_ym', ascending=True)
group = group.merge(prj50m_pt, how="left", on="id")
gdf50 = gpd.GeoDataFrame(group, geometry='geometry', crs=5179)
gdf100 = gpd.sjoin(prj100m, gdf50, how="inner", predicate="intersects")
results = round(gdf100.groupby(['std_ym', 'grid_100m', '주중주말'], as_index=False)['sum_pop'].sum())
results = pd.merge(results, prj100m, how="left", on="grid_100m")
results = gpd.GeoDataFrame(results, geometry="geometry")
results.columns = ['기준연월', 'grid_100m', '주중주말', '인구수', 'geometry']
results.to_file(data_path + r"02_집계결과\\대중교통 이용현황 집계결과.gpkg", layer="월별 100m 주중주말 인구현황", driver="GPKG")
results.to_csv(data_path + r"02_집계결과\\월별 100m 주중주말 인구현황.csv", sep=",", encoding="utf-8", index=False)


In [59]:
import pandas as pd
import geopandas as gpd
import numpy as np
import json
import plotly.graph_objects as go
from shapely import wkt
import pyproj

def build_population_map(csv_path, output_path, vworld_key):

    df = pd.read_csv(csv_path)
    df['geometry'] = df['geometry'].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:5179')
    gdf = gdf.to_crs('EPSG:4326')
    gdf['연도'] = gdf['기준연월'].str[:4]
    gdf['월'] = gdf['기준연월'].str[5:7] + '월'

    # ── 격자 중심 좌표 변환 ───────────────────────────────────
    gdf_proj = gdf.to_crs('EPSG:5179')
    gdf['_cx'] = gdf_proj.geometry.centroid.x
    gdf['_cy'] = gdf_proj.geometry.centroid.y
    transformer = pyproj.Transformer.from_crs('EPSG:5179', 'EPSG:4326', always_xy=True)
    lons, lats = transformer.transform(gdf['_cx'].values, gdf['_cy'].values)
    gdf['lon'] = lons
    gdf['lat'] = lats

    map_center = [gdf['lat'].mean(), gdf['lon'].mean()]

    # ── 히트맵용: 연도별 주중주말별 격자 평균 (연간합/12) ──────
    heatmap_df = gdf.groupby(['연도','주중주말','grid_100m','lat','lon'])['인구수'].sum().reset_index()
    heatmap_df['월평균'] = (heatmap_df['인구수'] / 12).astype(int)

    # ── 격자 폴리곤 좌표 ──────────────────────────────────────
    grid_polygons = {}
    for _, row in gdf[['grid_100m','geometry']].drop_duplicates('grid_100m').iterrows():
        coords = list(row['geometry'].exterior.coords)
        grid_polygons[row['grid_100m']] = [[c[1], c[0]] for c in coords]

    # ── 시계열용 ──────────────────────────────────────────────
    월순서 = ['01월','02월','03월','04월','05월','06월','07월','08월','09월','10월','11월','12월']
    ts_df = gdf.groupby(['연도','주중주말','월'])['인구수'].sum().reset_index()

    연도목록 = sorted(gdf['연도'].unique())
    연도색상 = {'2023': '#4169E1', '2024': '#DC143C', '2025': '#228B22'}
    구분색상 = {'주중': '#4169E1', '주말': '#DC143C'}

    # ── 차트 데이터 JSON (JS에서 직접 그리기) ────────────────
    chart_json = {}
    for 연도 in 연도목록:
        chart_json[연도] = {}
        for 구분 in ['주중', '주말']:
            tdf = ts_df[(ts_df['연도'] == 연도) & (ts_df['주중주말'] == 구분)].copy()
            tdf = tdf.set_index('월').reindex(월순서).reset_index()
            chart_json[연도][구분] = {
                'x': 월순서,
                'y': [float(v) if not np.isnan(v) else 0 for v in tdf['인구수'].values],
                'color': 구분색상[구분],
            }

    # ── 히트맵 JSON ───────────────────────────────────────────
    heatmap_json = {}
    for 연도 in 연도목록:
        heatmap_json[연도] = {}
        for 구분 in ['주중', '주말']:
            hdf = heatmap_df[(heatmap_df['연도'] == 연도) & (heatmap_df['주중주말'] == 구분)]
            max_val = hdf['월평균'].max()
            grids = []
            for _, r in hdf.iterrows():
                val = int(r['월평균'])
                norm = val / max_val if max_val > 0 else 0
                grids.append({
                    'grid_id': r['grid_100m'],
                    'val': val,
                    'norm': round(norm, 3),
                    'coords': grid_polygons.get(r['grid_100m'], []),
                    'lat': r['lat'],
                    'lon': r['lon'],
                })
            heatmap_json[연도][구분] = grids

    # ── HTML 생성 ─────────────────────────────────────────────
    html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>다대포해수욕장 인구 현황</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Noto Sans KR', sans-serif; background: #f0f2f5; }}
  h1 {{ text-align: center; padding: 14px; font-size: 17px; background: white;
       border-bottom: 1px solid #ddd; color: #333; }}
  .year-tabs {{ display: flex; justify-content: center; gap: 8px; padding: 10px;
               background: white; border-bottom: 1px solid #ddd; }}
  .tab-btn {{ padding: 6px 28px; border: 2px solid #ccc; border-radius: 20px;
              cursor: pointer; font-size: 14px; background: white;
              color: #333; transition: all 0.2s; font-weight: bold; }}
  .tab-btn.active {{ color: white; border-color: transparent; }}
  .main {{ display: flex; gap: 10px; padding: 10px; height: calc(100vh - 105px); }}
  .maps-col {{ display: flex; flex-direction: column; gap: 10px; flex: 1; min-width: 0; }}
  .map-wrap {{ flex: 1; border-radius: 8px; overflow: hidden;
              box-shadow: 0 2px 8px rgba(0,0,0,0.15); position: relative; }}
  .map-title {{ position: absolute; top: 8px; left: 50%; transform: translateX(-50%);
               z-index: 999; background: rgba(255,255,255,0.92); padding: 3px 14px;
               border-radius: 14px; font-size: 12px; font-weight: bold;
               box-shadow: 0 1px 4px rgba(0,0,0,0.2); white-space: nowrap; }}
  .chart-col {{ flex: 1; min-width: 0; background: white; border-radius: 8px;
               box-shadow: 0 2px 8px rgba(0,0,0,0.15); overflow: hidden;
               display: flex; align-items: center; justify-content: center; }}
  #bar-chart {{ width: 100%; height: 100%; }}
  .leaflet-map {{ width: 100%; height: 100%; }}
</style>
</head>
<body>
<h1>다대포해수욕장 월별 100m 격자 인구 현황</h1>
<div class="year-tabs">
  {''.join(f'<button class="tab-btn" id="tab-{y}" onclick="switchYear(\'{y}\')">{y}년</button>' for y in 연도목록)}
</div>
<div class="main">
  <div class="maps-col">
    <div class="map-wrap">
      <div class="map-title" id="title-주중" style="color:#4169E1;">주중 (월평균)</div>
      <div id="map-주중" class="leaflet-map"></div>
    </div>
    <div class="map-wrap">
      <div class="map-title" id="title-주말" style="color:#DC143C;">주말 (월평균)</div>
      <div id="map-주말" class="leaflet-map"></div>
    </div>
  </div>
  <div class="chart-col">
    <div id="bar-chart"></div>
  </div>
</div>

<script>
const VWORLD_KEY = "{vworld_key}";
const heatmapData = {json.dumps(heatmap_json, ensure_ascii=False)};
const chartData = {json.dumps(chart_json, ensure_ascii=False)};
const mapCenter = [{map_center[0]}, {map_center[1]}];
const yearColors = {json.dumps(연도색상, ensure_ascii=False)};
const typeColors = {{ '주중': '#4169E1', '주말': '#DC143C' }};

// ── 지도 초기화 ───────────────────────────────────────────
const maps = {{}};
const layers = {{ '주중': [], '주말': [] }};

['주중','주말'].forEach(type => {{
  maps[type] = L.map('map-' + type, {{ zoomControl: type === '주중' }})
               .setView(mapCenter, 17);
  L.tileLayer(
    `https://api.vworld.kr/req/wmts/1.0.0/${{VWORLD_KEY}}/Base/{{z}}/{{y}}/{{x}}.png`,
    {{ attribution: '브이월드', maxZoom: 19 }}
  ).addTo(maps[type]);

  maps[type].on('moveend zoomend', function() {{
    const other = type === '주중' ? '주말' : '주중';
    if (!maps[other]._syncing) {{
      maps[type]._syncing = true;
      maps[other].setView(maps[type].getCenter(), maps[type].getZoom(), {{animate: false}});
      maps[type]._syncing = false;
    }}
  }});
}});

// ── 히트맵 그리기 ─────────────────────────────────────────
function hexToRgb(hex) {{
  return {{
    r: parseInt(hex.slice(1,3),16),
    g: parseInt(hex.slice(3,5),16),
    b: parseInt(hex.slice(5,7),16)
  }};
}}

function drawHeatmap(year, type) {{
  layers[type].forEach(l => maps[type].removeLayer(l));
  layers[type] = [];
  const grids = heatmapData[year][type];
  const rgb = hexToRgb(typeColors[type]);

  grids.forEach(g => {{
    if (!g.coords || g.coords.length === 0) return;
    const alpha = 0.2 + g.norm * 0.6;
    const fillColor = `rgba(${{rgb.r}},${{rgb.g}},${{rgb.b}},${{alpha.toFixed(2)}})`;
    const borderColor = `rgba(${{rgb.r}},${{rgb.g}},${{rgb.b}},0.9)`;

    const poly = L.polygon(g.coords, {{
      color: borderColor, weight: 1,
      fillColor: fillColor, fillOpacity: 1,
    }}).addTo(maps[type]);
    poly.bindTooltip(`${{g.grid_id}}<br>월평균: ${{g.val.toLocaleString()}}명`, {{sticky: true}});
    layers[type].push(poly);

    const textColor = g.norm > 0.45 ? 'white' : '#222';
    const label = L.marker([g.lat, g.lon], {{
      icon: L.divIcon({{
        className: '',
        html: `<div style="font-size:10px;font-weight:bold;color:${{textColor}};
                           text-align:center;white-space:nowrap;
                           text-shadow:0 0 3px rgba(0,0,0,0.5);">
                 ${{g.val.toLocaleString()}}
               </div>`,
        iconAnchor: [22, 8],
        iconSize: [44, 16],
      }})
    }}).addTo(maps[type]);
    layers[type].push(label);
  }});
}}

// ── 바 차트 그리기 (Plotly JS 직접 호출) ─────────────────
function drawBarChart(year) {{
  const data = chartData[year];
  const traces = ['주중', '주말'].map(type => ({{
    type: 'bar',
    x: data[type].x,
    y: data[type].y,
    name: type,
    marker: {{ color: data[type].color }},
    opacity: 0.8,
    hovertemplate: type + ' %{{x}}: %{{y:,.0f}}명<extra></extra>',
  }}));

  const layout = {{
    barmode: 'group',
    title: {{ text: year + '년 주중/주말 월별 인구수', font: {{ size: 13 }} }},
    margin: {{ l: 60, r: 20, t: 50, b: 50 }},
    plot_bgcolor: 'white',
    paper_bgcolor: 'white',
    xaxis: {{
      title: '월',
      showgrid: false,
      categoryorder: 'array',
      categoryarray: {json.dumps(월순서)},
    }},
    yaxis: {{
      title: '인구수(명)',
      showgrid: true,
      gridcolor: '#dddddd',
    }},
    legend: {{ x: 0.01, y: 0.99 }},
  }};

  Plotly.react('bar-chart', traces, layout, {{responsive: true}});
}}

// ── 연도 전환 ─────────────────────────────────────────────
function switchYear(year) {{
  document.querySelectorAll('.tab-btn').forEach(b => {{
    b.classList.remove('active');
    b.style.background = 'white';
    b.style.color = '#333';
  }});
  const btn = document.getElementById('tab-' + year);
  btn.classList.add('active');
  btn.style.background = yearColors[year];

  document.getElementById('title-주중').textContent = year + '년 주중 (월평균)';
  document.getElementById('title-주말').textContent = year + '년 주말 (월평균)';

  drawHeatmap(year, '주중');
  drawHeatmap(year, '주말');
  drawBarChart(year);

  setTimeout(() => {{
    maps['주중'].invalidateSize();
    maps['주말'].invalidateSize();
  }}, 100);
}}

switchYear('{연도목록[0]}');
</script>
</body>
</html>"""

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"완료! → {output_path}")

# ── 실행 ──────────────────────────────────────────────────
build_population_map(
    csv_path=data_path + r'02_집계결과\\월별 100m 주중주말 인구현황.csv',
    output_path=data_path + r'03_시각화결과\\다대포_인구현황3.html',
    vworld_key='326A4FCA-0A3C-3CD1-83FD-A30126D77357'
)

완료! → D:\02_사업관리\2026년\데이터 분석 컨설팅\20260108_도시공공디자인담당관\\03_시각화결과\\다대포_인구현황3.html
